# Exercise 02 - Build the evidence: a deterministic, anchored synthetic ledger (SG-1)

**Project Aegis - Phase 1 (Data & Storage) - learning loop step 2**

### Why this exercise
Exercise 01 gave you the **claim** (what JPMorgan asserts). Now we cross the public/private waterline
to the **evidence** side - the transaction-level ledger. For a public company that ledger is never
public (it's MNPI), so we **simulate** it with Faker (spec FR-IN-2) - but anchored to the *real*
claimed number so the two can be reconciled. Then you'll write the reconciliation that flags a delta.
That comparison **is** SG-1: *given a filing claiming $X and a ledger summing to $Y != $X, flag it.*

### What you'll learn
- **Money is `Decimal`, never `float`** - the single most important data-correctness rule in finance.
- **Anchored synthetic generation** - make N transactions sum *exactly* to a target (the remainder fix-up).
- **Determinism / reproducibility** (NFR-CORR-1) - same seed -> byte-identical ledger, every run.
- **The seeded discrepancy** - planting a known landmine so we own the ground truth.
- **Reconciliation as a `diff`** - the audit operation at the heart of the system.

### Requirements
- `pip install faker` (the one dependency).
- Internet is *optional* here - we reuse Exercise 01 to pull the live claim, with an offline fallback.

## Setup - get a claim to anchor to

We reuse the Exercise 01 connector (given here) to pull JPMorgan's latest reported `Liabilities`. If
you're offline, it falls back to the value we recorded earlier so the rest of the notebook still runs.

In [1]:
import urllib.request, json, random
from decimal import Decimal
from faker import Faker

SEC_USER_AGENT = {"User-Agent": "Aegis Learning Project sunitsingh.bitsg@gmail.com"}

def _pull_claim_live(ticker, concept="Liabilities"):
    def get(u):
        return json.load(urllib.request.urlopen(urllib.request.Request(u, headers=SEC_USER_AGENT), timeout=60))
    tbl = get("https://www.sec.gov/files/company_tickers.json")
    cik = next(str(r["cik_str"]).zfill(10) for r in tbl.values() if r["ticker"].upper() == ticker.upper())
    facts = get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")
    rec = max((e for e in facts["facts"]["us-gaap"][concept]["units"]["USD"] if e.get("form") == "10-K"),
              key=lambda e: e["end"])
    return {"entity": facts["entityName"], "cik": cik, "concept": concept,
            "claimed_value": rec["val"], "period_end": rec["end"],
            "provenance": {"form": rec["form"], "fy": rec.get("fy"), "accession": rec["accn"]}}

try:
    claim_record = _pull_claim_live("JPM")
    print("pulled live claim")
except Exception as e:
    print("offline, using recorded claim:", e)
    claim_record = {"entity": "JPMORGAN CHASE & CO", "cik": "0000019617", "concept": "Liabilities",
                    "claimed_value": 4062462000000, "period_end": "2025-12-31",
                    "provenance": {"form": "10-K", "fy": 2025, "accession": "0001628280-26-008131"}}

CLAIMED = Decimal(claim_record["claimed_value"])   # exact, from the integer cents-free USD value
print(claim_record["entity"], claim_record["concept"], "=", f"${CLAIMED:,}")

pulled live claim
JPMORGAN CHASE & CO Liabilities = $4,062,462,000,000


## Money is `Decimal`, never `float`

Floats are base-2 and cannot represent most decimal fractions exactly. Summing thousands of float
dollar amounts accumulates error - and in an audit, a one-cent drift is a *failed reconciliation*.
Run this to see the trap, then note we use `Decimal` everywhere money appears.

In [2]:
print("float :", 0.1 + 0.2, "  equals 0.3?", (0.1 + 0.2) == 0.3)
print("Decimal:", Decimal('0.1') + Decimal('0.2'), "equals 0.3?", (Decimal('0.1') + Decimal('0.2')) == Decimal('0.3'))

float : 0.30000000000000004   equals 0.3? False
Decimal: 0.3 equals 0.3? True


## Step 1 - Generate a ledger that sums *exactly* to a target

The evidence ledger must reconcile to the claim, so its amounts have to sum to a chosen target **to
the cent**. The robust technique (used in real allocation/billing systems):

1. Work in **integer cents**, never dollars - integers don't drift.
2. Split the target across `n` rows by random weights.
3. Rounding each row loses a few cents, so the parts won't sum to the target. **Dump the leftover
   remainder onto the last row** - now the sum is exact by construction.

Faker fills realistic metadata (counterparties, dates); seeding both `random` and `Faker` makes the
whole ledger **reproducible** (NFR-CORR-1). *(Double-entry note: a real GL stores each amount twice,
as a debit and a credit that net to zero; here we model the single liability account's rows whose
magnitudes reconcile to the claim - the same `SUM`-vs-claim idea, minus the bookkeeping plumbing.)*

**Your task (gap 1):** write the one line that applies the **remainder fix-up** so the rows sum exactly.

In [10]:
def make_ledger(target_total: Decimal, n: int, seed: int,
                account: str = "operating_lease_liability") -> list:
    """Generate n synthetic transactions whose amounts sum EXACTLY to target_total."""
    rng = random.Random(seed)
    fake = Faker(); fake.seed_instance(seed)

    target_cents = int((target_total * 100).to_integral_value())
    weights = [rng.random() for _ in range(n)]
    wsum = sum(weights)
    cents = [int(target_cents * w / wsum) for w in weights]   # rounding loses a few cents

    # ==== YOUR CODE HERE (start) ====
    # The parts don't sum to target_cents yet. Add the leftover (target_cents - sum(cents))
    # to the last row so the totals match exactly. One line.
    cents[-1] += target_cents - sum(cents)
    # raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====

    rows = []
    for i in range(n):
        rows.append({
            "txn_id": f"L-{seed}-{i:05d}",
            "date": fake.date_between(start_date='-1y').isoformat(),
            "counterparty": fake.company(),
            "account": account,
            "amount": Decimal(cents[i]) / 100,
        })
    return rows

def ledger_total(rows) -> Decimal:
    return sum((r["amount"] for r in rows), Decimal("0"))

# --- self-check: exact sum + determinism ---
a = make_ledger(CLAIMED, 500, seed=42)
assert ledger_total(a) == CLAIMED, f"should sum to claim exactly, off by {CLAIMED - ledger_total(a)}"
b = make_ledger(CLAIMED, 500, seed=42)
assert [r['amount'] for r in a] == [r['amount'] for r in b], "same seed must reproduce the ledger"
assert [r['counterparty'] for r in a] == [r['counterparty'] for r in b], "Faker must be seeded too"
print("PASS - 500 rows sum to exactly", f"${ledger_total(a):,}", "and are reproducible")
print("  e.g.", a[0])

PASS - 500 rows sum to exactly $4,062,462,000,000.00 and are reproducible
  e.g. {'txn_id': 'L-42-00000', 'date': '2026-03-02', 'counterparty': 'Johnson LLC', 'account': 'operating_lease_liability', 'amount': Decimal('10360701733.61')}


## Step 2 - Plant a seeded discrepancy (a known landmine)

A *clean* book is anchored to the claim (sums to `$X`). To test detection, we generate a book that
sums to a deliberately different total `$Y` - modeling an **understated liability** (the real
obligations differ from what the filing claims). Because *we* chose the delta, we own the ground
truth - which is what makes evaluation possible later (spec section 11.6). This cell is given.

In [11]:
clean_ledger = make_ledger(CLAIMED, n=500, seed=42)

SEEDED_DELTA = Decimal("125000000.00")   # a $125M understatement we deliberately introduce
discrepant_ledger = make_ledger(CLAIMED - SEEDED_DELTA, n=500, seed=7)

print("clean book sums to     :", f"${ledger_total(clean_ledger):,}")
print("discrepant book sums to:", f"${ledger_total(discrepant_ledger):,}")
print("planted delta          :", f"${CLAIMED - ledger_total(discrepant_ledger):,}")

clean book sums to     : $4,062,462,000,000.00
discrepant book sums to: $4,062,337,000,000.00
planted delta          : $125,000,000.00


## Step 3 - Reconcile: the audit operation (gap 2)

Reconciliation is just a `diff` between the claim and the evidence: sum the ledger, subtract from the
claimed value, and classify. Because everything is `Decimal`, a clean book reconciles to **exactly**
zero - no tolerance hand-waving. This function is what the Financial agent will later call as a
parameterized SQL `SUM` (Exercise 03), and its output is the seed of the four-tier audit trail.

**Your task (gap 2):** finish `reconcile` - compute `ledger_total`, the `delta = claimed - ledger`,
and set `status` to `"MATCH"` when delta is zero else `"DISCREPANCY"`.

In [15]:
def reconcile(claim_record: dict, ledger: list) -> dict:
    """Compare a claim against ledger evidence; flag any delta as a finding."""
    claimed = Decimal(claim_record["claimed_value"])
    # ==== YOUR CODE HERE (start) ====
    # 1) total = sum of the ledger amounts (use ledger_total)
    total = sum([statement['amount'] for statement in ledger])
    # 2) delta = claimed - total
    delta = claim_record['claimed_value'] - total
    # 3) status = 'MATCH' if delta == 0 else 'DISCREPANCY'
    if delta == 0:
        status = 'MATCH'
    else:
        status = 'DISCREPANCY'
    # raise NotImplementedError
    # ==== YOUR CODE HERE (end) ====
    return {
        "entity": claim_record["entity"],
        "concept": claim_record["concept"],
        "claimed_value": claimed,
        "ledger_total": total,
        "delta": delta,
        "status": status,
        "claim_provenance": claim_record["provenance"],
        "evidence_rows": len(ledger),
    }

# --- self-check ---
clean_finding = reconcile(claim_record, clean_ledger)
assert clean_finding["status"] == "MATCH" and clean_finding["delta"] == Decimal("0"), clean_finding
disc_finding = reconcile(claim_record, discrepant_ledger)
assert disc_finding["status"] == "DISCREPANCY", disc_finding
assert disc_finding["delta"] == SEEDED_DELTA, f"expected {SEEDED_DELTA}, got {disc_finding['delta']}"
print("PASS - clean -> MATCH (delta 0); discrepant -> DISCREPANCY (delta", f"${disc_finding['delta']:,})")

PASS - clean -> MATCH (delta 0); discrepant -> DISCREPANCY (delta $125,000,000.00)


## Step 4 - The finding (seed of the audit trail)

This is SG-1 end to end: a real claim, anchored synthetic evidence, and a reconciliation that flags
the delta and **cites both sides**. In the full system the claim's `accession` and the ledger's SQL
become the *Grounding* tier of the four-tier audit trail. Run it to see the finding.

In [16]:
def render_finding(f: dict) -> str:
    pct = (f["delta"] / f["claimed_value"] * 100) if f["claimed_value"] else Decimal(0)
    lines = [
        f"FINDING [{f['status']}] - {f['entity']} / {f['concept']}",
        f"  claimed (10-K)   : ${f['claimed_value']:,}  (accn {f['claim_provenance']['accession']})",
        f"  ledger evidence  : ${f['ledger_total']:,}  ({f['evidence_rows']} txns)",
        f"  delta            : ${f['delta']:,}  ({pct:.4f}% of claim)",
    ]
    return "\n".join(lines)

print(render_finding(disc_finding))

FINDING [DISCREPANCY] - JPMORGAN CHASE & CO / Liabilities
  claimed (10-K)   : $4,062,462,000,000  (accn 0001628280-26-008131)
  ledger evidence  : $4,062,337,000,000.00  (500 txns)
  delta            : $125,000,000.00  (0.0031% of claim)


## Recap & what's next

You built the **evidence** side and the **reconciliation** - SG-1 in miniature:
- money as `Decimal` (correctness by construction),
- an **anchored** synthetic ledger that sums exactly to a real claim, **reproducible** from a seed,
- a **seeded discrepancy** giving owned ground truth,
- reconciliation as a `diff` that flags the delta and cites both sides.

**Exercise 03** moves this ledger out of memory and into **PostgreSQL** - the real evidence store -
and replaces `ledger_total()` with a **parameterized SQL `SUM`**, which is what the Postgres MCP tool
(spec FR-MCP-1) exposes to the agents. Same audit, real database, real SQL.

**Stretch goals (optional):**
1. Make the discrepancy a *percentage* parameter instead of an absolute amount.
2. Add a `severity` field (e.g. by |delta|/claim crossing thresholds).
3. **Benford's Law:** check the leading-digit distribution of your amounts - real ledgers follow it,
   and deviation is a classic fraud signal. (Our uniform-random amounts will *not* follow it - a nice
   lesson in why 'realistic' synthetic data is harder than it looks.)